In [ ]:
# 1. Import libraries
from pathlib import Path

import rasterio
from rasterio.warp import Resampling, calculate_default_transform, reproject


In [ ]:
# 2. Set parameters
INPUT_RASTER = Path("merged_pnw_firedv2_bccbi.tif")
OUTPUT_DIR = Path(".")
START_YEAR = 2012
END_YEAR = 2024
TARGET_CRS = "EPSG:5070"
TARGET_RES = 375

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 3. Helper: map year -> source band
def build_band_lookup(src):
    """Map year to source band index using names like year_2012."""
    lookup = {}
    for i, name in enumerate(src.descriptions, start=1):
        if not name:
            continue
        if name.startswith("year_"):
            try:
                year = int(name.split("_")[1])
                lookup[year] = i
            except (IndexError, ValueError):
                continue
    return lookup


In [ ]:
# 4. Open source raster and build target grid
with rasterio.open(INPUT_RASTER) as src:
    band_lookup = build_band_lookup(src)

    transform, width, height = calculate_default_transform(
        src.crs,
        TARGET_CRS,
        src.width,
        src.height,
        *src.bounds,
        resolution=TARGET_RES,
    )

    print(f"Source CRS: {src.crs}")
    print(f"Target CRS: {TARGET_CRS}")
    print(f"Output grid: {width} x {height} @ {TARGET_RES} m")


In [ ]:
# 5. Export annual CBI rasters (bilinear resampling)
with rasterio.open(INPUT_RASTER) as src:
    band_lookup = build_band_lookup(src)

    transform, width, height = calculate_default_transform(
        src.crs,
        TARGET_CRS,
        src.width,
        src.height,
        *src.bounds,
        resolution=TARGET_RES,
    )

    for year in range(START_YEAR, END_YEAR + 1):
        if year not in band_lookup:
            print(f"⚠️ Year {year} not found in source band descriptions; skipping.")
            continue

        band_index = band_lookup[year]
        cbi_data = src.read(band_index)
        out_path = OUTPUT_DIR / f"{year}_pnw_firedv2_bccbi_epsg5070.tif"

        meta = src.meta.copy()
        meta.update(
            {
                "count": 1,
                "crs": TARGET_CRS,
                "transform": transform,
                "width": width,
                "height": height,
                "nodata": src.nodata,
            }
        )

        with rasterio.open(out_path, "w", **meta) as dst:
            reproject(
                source=cbi_data,
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=TARGET_CRS,
                resampling=Resampling.bilinear,
                src_nodata=src.nodata,
                dst_nodata=src.nodata,
            )
            dst.set_band_description(1, f"CBI_{year}")

        print(f"✅ Saved: {out_path}")
